# Image Comparison Analysis - Bone Scan

This notebook compares two images and analyzes the similarity of specific color regions (red and blue).

## Setup for Google Colab

If running on Google Colab, uncomment and run the following cell to upload your images:

In [ ]:
# Google Colab用のファイルアップロード
# import os
# from google.colab import files

# # アップロードされたファイルを確認
# print("Please upload your image files (origin.png and filter.png)")
# uploaded = files.upload()

# # アップロードされたファイル名を表示
# for filename in uploaded.keys():
#     print(f'Uploaded: {filename}')

## Import necessary libraries

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from scipy.spatial.distance import directed_hausdorff
import os

## Define utility functions

In [ ]:
def calculate_dice_coefficient(m1, m2):
    """Calculate Dice coefficient for two binary masks"""
    if m1.sum() + m2.sum() == 0:
        return 1.0  # 両方空なら完全一致
    return 2.0 * np.logical_and(m1, m2).sum() / (m1.sum() + m2.sum())

def calculate_hausdorff_distance(m1, m2):
    """Calculate Hausdorff distance between two binary masks"""
    c1 = np.column_stack(np.where(m1))
    c2 = np.column_stack(np.where(m2))
    if c1.size == 0 or c2.size == 0:
        return np.nan  # どちらか空なら距離 Undefined
    return max(directed_hausdorff(c1, c2)[0],
               directed_hausdorff(c2, c1)[0])

def segment_color_regions(img, lower, upper):
    """Segment color regions based on HSV range"""
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    return cv2.inRange(hsv, lower, upper)

def create_overlay_image_with_white_background(img_ref, m1, m2, alpha=0.25):
    """Create overlay image showing matching and non-matching regions"""
    white_bg = np.ones_like(img_ref) * 255
    overlay = np.zeros_like(img_ref)
    overlay[np.logical_and(m1, m2)] = [0, 255, 0]   # green (matching)
    overlay[np.logical_xor(m1, m2)] = [255, 0, 0]   # red (non-matching)
    return cv2.addWeighted(white_bg, alpha, overlay, 1-alpha, 0)

def align_images_by_contours(img1, img2):
    """Align two images based on their largest contours"""
    g1 = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
    g2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)
    _, t1 = cv2.threshold(g1, 0, 255, cv2.THRESH_OTSU)
    _, t2 = cv2.threshold(g2, 0, 255, cv2.THRESH_OTSU)
    cnt1, _ = cv2.findContours(t1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnt2, _ = cv2.findContours(t2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    x1, y1, w1, h1 = cv2.boundingRect(max(cnt1, key=cv2.contourArea))
    x2, y2, w2, h2 = cv2.boundingRect(max(cnt2, key=cv2.contourArea))
    crop1 = img1[y1:y1+h1, x1:x1+w1]
    crop2 = cv2.resize(img2[y2:y2+h2, x2:x2+w2], (w1, h1))
    return crop1, crop2

## Set image paths

Update these paths to match your image files. For Google Colab, use the uploaded file names.

In [ ]:
# 画像ファイルのパスを設定
# デフォルトでは同じフォルダ内のorigin.pngとfilter.pngを使用
img1_path = "origin.png"
img2_path = "filter.png"

# ファイルの存在確認
if not os.path.exists(img1_path):
    print(f"Warning: {img1_path} not found. Please update the path or upload the file.")
if not os.path.exists(img2_path):
    print(f"Warning: {img2_path} not found. Please update the path or upload the file.")

# 利用可能なPNGファイルを表示
png_files = [f for f in os.listdir('.') if f.endswith('.png')]
if png_files:
    print("Available PNG files in current directory:")
    for f in png_files:
        print(f"  - {f}")

## Load and align images

In [ ]:
# 画像を読み込み
img1 = imread(img1_path)
img2 = imread(img2_path)

# 画像のサイズを表示
print(f"Image 1 shape: {img1.shape}")
print(f"Image 2 shape: {img2.shape}")

# 画像を位置合わせ
aligned1, aligned2 = align_images_by_contours(img1, img2)
print(f"Aligned shape: {aligned1.shape}")

## Display original and aligned images

In [ ]:
# 元画像と位置合わせ後の画像を表示
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(img1)
axes[0, 0].set_title("Original Image 1 (Origin)")
axes[0, 0].axis('off')

axes[0, 1].imshow(img2)
axes[0, 1].set_title("Original Image 2 (Filter)")
axes[0, 1].axis('off')

axes[1, 0].imshow(aligned1)
axes[1, 0].set_title("Aligned Image 1")
axes[1, 0].axis('off')

axes[1, 1].imshow(aligned2)
axes[1, 1].set_title("Aligned Image 2")
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

## Define HSV color ranges for red and blue regions

In [ ]:
# HSV色範囲の定義
# 赤色の範囲（赤は色相が0度と180度付近にあるため2つの範囲を定義）
lower_red1 = np.array([0, 70, 50])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([170, 70, 50])
upper_red2 = np.array([180, 255, 255])

# 青色の範囲
lower_blue = np.array([100, 150, 0])
upper_blue = np.array([140, 255, 255])

## Segment color regions

In [ ]:
# 色領域のセグメンテーション
red1 = segment_color_regions(aligned1, lower_red1, upper_red1) | \
       segment_color_regions(aligned1, lower_red2, upper_red2)
red2 = segment_color_regions(aligned2, lower_red1, upper_red1) | \
       segment_color_regions(aligned2, lower_red2, upper_red2)

blue1 = segment_color_regions(aligned1, lower_blue, upper_blue)
blue2 = segment_color_regions(aligned2, lower_blue, upper_blue)

# セグメンテーション結果の統計情報
print(f"Red pixels in image 1: {np.sum(red1 > 0)}")
print(f"Red pixels in image 2: {np.sum(red2 > 0)}")
print(f"Blue pixels in image 1: {np.sum(blue1 > 0)}")
print(f"Blue pixels in image 2: {np.sum(blue2 > 0)}")

## Calculate similarity metrics

In [ ]:
# 類似度の計算
dice_r = calculate_dice_coefficient(red1 > 0, red2 > 0)
dice_b = calculate_dice_coefficient(blue1 > 0, blue2 > 0)
haus_r = calculate_hausdorff_distance(red1 > 0, red2 > 0)
haus_b = calculate_hausdorff_distance(blue1 > 0, blue2 > 0)

# 結果の表示
print("=" * 50)
print("Similarity Metrics:")
print("=" * 50)
print(f"Red regions:")
print(f"  - Dice coefficient: {dice_r:.3f}")
print(f"  - Hausdorff distance: {haus_r:.1f} pixels" if not np.isnan(haus_r) else "  - Hausdorff distance: N/A (empty region)")
print(f"\nBlue regions:")
print(f"  - Dice coefficient: {dice_b:.3f}")
print(f"  - Hausdorff distance: {haus_b:.1f} pixels" if not np.isnan(haus_b) else "  - Hausdorff distance: N/A (empty region)")
print("=" * 50)

## Visualize results

In [ ]:
# 結果の可視化
fig, ax = plt.subplots(1, 2, figsize=(14, 7))

# 赤色領域のオーバーレイ
ax[0].imshow(create_overlay_image_with_white_background(aligned1, red1 > 0, red2 > 0))
ax[0].set_title(f"Red Regions Overlay\nDice: {dice_r:.3f}, HD: {haus_r:.1f}px" if not np.isnan(haus_r) else f"Red Regions Overlay\nDice: {dice_r:.3f}, HD: N/A")
ax[0].axis('off')

# 青色領域のオーバーレイ
ax[1].imshow(create_overlay_image_with_white_background(aligned1, blue1 > 0, blue2 > 0))
ax[1].set_title(f"Blue Regions Overlay\nDice: {dice_b:.3f}, HD: {haus_b:.1f}px" if not np.isnan(haus_b) else f"Blue Regions Overlay\nDice: {dice_b:.3f}, HD: N/A")
ax[1].axis('off')

# 凡例の追加
fig.text(0.5, 0.02, 'Green: Matching regions | Red: Non-matching regions', 
         ha='center', fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## Save results (optional)

In [ ]:
# 結果を保存する場合は以下のコードをアンコメント
# output_red = create_overlay_image_with_white_background(aligned1, red1 > 0, red2 > 0)
# output_blue = create_overlay_image_with_white_background(aligned1, blue1 > 0, blue2 > 0)

# cv2.imwrite('red_comparison.png', cv2.cvtColor(output_red, cv2.COLOR_RGB2BGR))
# cv2.imwrite('blue_comparison.png', cv2.cvtColor(output_blue, cv2.COLOR_RGB2BGR))
# print("Results saved as 'red_comparison.png' and 'blue_comparison.png'")

# Google Colabの場合、ファイルをダウンロード
# from google.colab import files
# files.download('red_comparison.png')
# files.download('blue_comparison.png')